# Dashboard exports: the derived tables behind the public dashboard

This notebook prepares the small tables that the dashboard needs and that the other notebooks do not already write. It is the only place where dashboard-only numbers are computed, so that the website never calculates results itself.

**Rules for everything written here**
- Derived results and the project's own registers only. **Never an LME series**: copper in the Story chart is the **World Bank** monthly series (adapted from World Bank Commodity Price Data, CC BY 4.0), deflated with US CPI (BLS, via FRED).
- Every table is written with an explicit list of columns, and the notebook stops if a column name belongs to the licensed LME series.
- Detail text of data checks that mention LME data is withheld (commercial data); only the LME-versus-World-Bank agreement figures (a correlation and a gap) stay.
- What goes into the website is chosen in `tools/export_site_data.py`; the audit (`tools/audit_public.py`) checks the result.

Run `python copper_database/build_db.py --raw .` first, then notebooks 01 to 03.

In [ ]:
import sqlite3
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

FORBIDDEN_COLUMNS = {"cash_usd_t", "cash_unofficial_usd_t", "three_month_usd_t", "three_month_unofficial_usd_t", "fifteen_month_usd_t", "dec_1_usd_t",
                     "dec_2_usd_t", "dec_3_usd_t", "inventory_t", "volume_as_exported", "copper_lme_cash_usd_t_avg",
                     "copper_lme_cash_usd_t_month_end", "copper_lme_cash_month_end_date"}
con = sqlite3.connect("../copper_database/copper.db")
rd = lambda sql: pd.read_sql(sql, con)
written = {}


def plain(df):
    """Site text uses no em or en dashes: an em dash becomes a comma, an en dash a hyphen."""
    df = df.copy()
    for c in df.columns:
        df[c] = df[c].map(lambda v: v.replace("\u2014", ", ").replace("\u2013", "-") if isinstance(v, str) else v)
    return df


def save(df, name):
    if name.startswith("res_dashboard_"):
        df = plain(df)
    bad = FORBIDDEN_COLUMNS & set(df.columns)
    assert not bad, f"{name}: LME series column(s) {bad} must never be exported"
    df.to_csv(f"../results/{name}.csv", index=False)
    df.to_sql(name, con, if_exists="replace", index=False)
    written[name] = (len(df), len(df.columns))


MINE = rd("SELECT value FROM mart_demand_context WHERE metric='world_mine_production_2025e_t'").value.iloc[0]

## 1. Story, guess 1: how much of copper's monthly moves does the dollar account for?
Monthly log changes, 2006-02 to 2026-08 for the broad dollar index (247 months) and 1999-02 on for the euro (331 months). "Account for" means the R squared of a simple regression: a statistical share of the ups and downs, not a cause. The direction counts use the months in which the dollar index rose or fell.

In [ ]:
ch = rd("SELECT month, copper_usd_logret_m AS c, dollar_index_logret_m AS d, usd_per_eur_logret_m AS e, us10y_chg_pp_m AS r FROM mart_monthly_changes")
di = ch.dropna(subset=["c", "d"])
eu = ch.dropna(subset=["c", "e"])
dr = ch.dropna(subset=["c", "d", "r"])


def r_squared(y, X):
    X = np.column_stack([np.ones(len(y))] + [np.asarray(x) for x in X])
    beta, *_ = np.linalg.lstsq(X, np.asarray(y), rcond=None)
    res = np.asarray(y) - X @ beta
    return 1 - (res @ res) / ((np.asarray(y) - np.mean(y)) @ (np.asarray(y) - np.mean(y)))


up, dn = di[di.d > 0], di[di.d < 0]
p0, p1 = f"{di.month.min()[:7]}", f"{di.month.max()[:7]}"
# Direction counts, one definition everywhere: "opposite" means strictly opposite signs. A month in which either monthly average did not
# change at all (an exact zero) counts as neither opposite nor same direction. Every count below comes from the same sample, so they add up.
n_months = len(di)
fell_when_up = int((up.c < 0).sum())     # copper fell in a month the dollar index rose
rose_when_dn = int((dn.c > 0).sum())     # copper rose in a month the dollar index fell
n_opposite = fell_when_up + rose_when_dn
n_same = int((((di.c > 0) & (di.d > 0)) | ((di.c < 0) & (di.d < 0))).sum())
n_unchanged = int(((di.c == 0) | (di.d == 0)).sum())
assert n_opposite + n_same + n_unchanged == n_months, (n_opposite, n_same, n_unchanged, n_months)
assert n_opposite == int((((di.c < 0) & (di.d > 0)) | ((di.c > 0) & (di.d < 0))).sum())
guess = pd.DataFrame([
    ("r2_dollar_index", "Share of copper's monthly moves the broad dollar index accounts for (R squared)", r_squared(di.c, [di.d]) * 100, "percent", n_months, p0, p1),
    ("r2_euro", "Same for the euro alone (euros per dollar)", r_squared(eu.c, [eu.e]) * 100, "percent", len(eu), eu.month.min()[:7], eu.month.max()[:7]),
    ("r2_dollar_index_plus_yield", "Broad dollar index plus the change in the 10-year US yield", r_squared(dr.c, [dr.d, dr.r]) * 100, "percent", len(dr), dr.month.min()[:7], dr.month.max()[:7]),
    ("months_total", "Months compared (monthly changes of copper and the broad dollar index)", n_months, "months", n_months, p0, p1),
    ("dollar_rose_months", "Months in which the broad dollar index rose", len(up), "months", n_months, p0, p1),
    ("copper_fell_when_dollar_rose_months", "Of those, months in which copper fell", fell_when_up, "months", len(up), p0, p1),
    ("copper_fell_when_dollar_rose", "Share of those months in which copper fell", fell_when_up / len(up) * 100, "percent", len(up), p0, p1),
    ("dollar_fell_months", "Months in which the broad dollar index fell", len(dn), "months", n_months, p0, p1),
    ("copper_rose_when_dollar_fell_months", "Of those, months in which copper rose", rose_when_dn, "months", len(dn), p0, p1),
    ("copper_rose_when_dollar_fell", "Share of those months in which copper rose", rose_when_dn / len(dn) * 100, "percent", len(dn), p0, p1),
    ("opposite_months", "Months in which copper and the dollar index moved in strictly opposite directions", n_opposite, "months", n_months, p0, p1),
    ("opposite_direction_share", "Share of months in which they moved in opposite directions", n_opposite / n_months * 100, "percent", n_months, p0, p1),
    ("same_direction_months", "Months in which they moved in the same direction", n_same, "months", n_months, p0, p1),
    ("unchanged_months", "Months in which one of the two monthly averages did not change at all (counted as neither)", n_unchanged, "months", n_months, p0, p1),
], columns=["fact_id", "label", "value", "unit", "months", "period_from", "period_to"])
guess["value"] = guess["value"].round(2)
# the same R squared is already in notebook 01 (regression 2: dollar index only); the two must agree
reg = pd.read_csv("../results/res_dollar_regressions.csv")
r2_nb01 = reg[reg["model"].str.startswith("2.")]["R squared"].iloc[0] * 100
assert abs(guess.loc[0, "value"] - r2_nb01) < 0.01, (guess.loc[0, "value"], r2_nb01)
save(guess, "res_story_guess_dollar")
print("saved", len(guess), "rows; R squared agrees with notebook 01")

## 2. Story, guess 2: is copper at a record high?
World Bank monthly copper price (a monthly average, USD per tonne), nominal, and deflated to **August 2026 US dollars with US CPI**. This is a US-dollar view: it says nothing about what copper cost in another currency. US CPI has no value for October 2025, so that month's real price is empty.
The data start in January 1960, so "record" means the highest since 1960.

In [ ]:
ser = rd("""SELECT p.date AS month, p.copper_usd_t AS nominal_usd_t, r.copper_real_usd_t_latest_prices AS real_usd_t_aug2026
            FROM v_price_monthly p LEFT JOIN v_price_real r ON r.date = p.date WHERE p.copper_usd_t IS NOT NULL ORDER BY p.date""")
ser["nominal_usd_t"] = ser["nominal_usd_t"].round(2)
ser["real_usd_t_aug2026"] = ser["real_usd_t_aug2026"].round(1)
save(ser, "res_story_copper_series")

real = ser.dropna(subset=["real_usd_t_aug2026"]).copy()
real["year"] = real.month.str[:4].astype(int)
now_n, now_r = ser.nominal_usd_t.iloc[-1], real.real_usd_t_aug2026.iloc[-1]
nrec = ser.loc[ser.nominal_usd_t.idxmax()]
rpk = real.loc[real.real_usd_t_aug2026.idxmax()]
r90 = real[real.year >= 1990].sort_values("real_usd_t_aug2026", ascending=False)
r90 = r90[r90.month != real.month.iloc[-1]].iloc[0]
r60 = real[real.year <= 1969].loc[lambda d: d.real_usd_t_aug2026.idxmax()]
facts = pd.DataFrame([
    ("series_months", ser.month.iloc[0], len(ser), "months", "World Bank monthly copper price, January 1960 to the latest month"),
    ("nominal_latest", ser.month.iloc[-1], now_n, "USD per tonne", "latest monthly average"),
    ("nominal_record", nrec.month, nrec.nominal_usd_t, "USD per tonne", "highest nominal monthly average in the series"),
    ("real_latest", real.month.iloc[-1], now_r, "USD per tonne, August 2026 dollars", "deflated with US CPI (a US-dollar view); the latest month is the base"),
    ("real_months_above_latest", real.month.iloc[-1], int((real.real_usd_t_aug2026 > now_r).sum()), "months", "months in which the real price was higher than the latest month"),
    ("real_peak_all", rpk.month, rpk.real_usd_t_aug2026, "USD per tonne, August 2026 dollars", "highest real monthly price since 1960"),
    ("real_peak_vs_latest_pct", rpk.month, (rpk.real_usd_t_aug2026 / now_r - 1) * 100, "percent", "how far above the latest month the real peak is"),
    ("real_peak_1960s", r60.month, r60.real_usd_t_aug2026, "USD per tonne, August 2026 dollars", "highest real price in the 1960s"),
    ("real_peak_since_1990", r90.month, r90.real_usd_t_aug2026, "USD per tonne, August 2026 dollars", "highest real price since 1990"),
    ("latest_vs_real_peak_since_1990_pct", real.month.iloc[-1], (now_r / r90.real_usd_t_aug2026 - 1) * 100, "percent", "the latest month against the real high since 1990"),
    ("cpi_gap", "2025-10-01", np.nan, "", "US CPI has no value for October 2025 (federal shutdown, BLS), so the real price for that month is empty"),
], columns=["fact_id", "month", "value", "unit", "note"])
facts["value"] = facts["value"].round(2)
assert nrec.month == ser.month.iloc[-1], "nominal record is no longer the latest month: the Story wording must change"
save(facts, "res_story_copper_record_facts")
print(facts[["fact_id", "month", "value"]].to_string(index=False))

## 3. Demand scenario: the case grids and the two ranges
The interactive selectors on the Demand page read these grids. The combined "change in copper in new cars" column is **not** exported, so the website can never present it as "the EV effect".

In [ ]:
evc = rd("SELECT * FROM mart_ev_copper_cases")
ev_out = evc[["iea_scenario", "year", "path_method", "is_interpolated", "bev_case", "bev_kg", "phev_case", "phev_kg", "petrol_case", "petrol_kg", "growth_case",
              "extra_bev", "extra_phev", "delta_total_sales", "transition_t", "market_growth_t", "transition_all_evs_t"]].copy()
for c in ["extra_bev", "extra_phev", "delta_total_sales", "transition_t", "market_growth_t", "transition_all_evs_t"]:
    ev_out[c] = ev_out[c].round(1)
save(ev_out, "res_demand_ev_cases")

dcx = rd("SELECT * FROM mart_dc_copper_cases")
dcx["is_slower_build"] = (dcx["build_in_target_year_gw"] < dcx["base_build_gw"]).astype(int)
dc_out = dcx[["iea_case", "basis", "year", "is_exploratory", "path", "intensity_case", "t_per_mw", "base_build_case", "build_in_target_year_gw", "base_build_gw",
              "new_capacity_copper_t", "base_year_copper_t", "extra_vs_base_t", "is_slower_build"]].copy()
for c in ["build_in_target_year_gw", "base_build_gw", "new_capacity_copper_t", "base_year_copper_t", "extra_vs_base_t"]:
    dc_out[c] = dc_out[c].round(3)
save(dc_out, "res_demand_dc_cases")

paths = rd("SELECT iea_scenario, path_method, year, is_interpolated, bev, phev, ev, share_pct, total_sales_central, total_sales_min, total_sales_max, non_ev_sales_central FROM mart_ev_sales_path")
save(paths.round(2), "res_demand_ev_paths")
cap = rd("SELECT iea_case, basis, year, capacity_gw, is_exploratory FROM stg_datacentre_capacity_scenarios")
save(cap, "res_demand_dc_capacity")
dcp = rd("""SELECT iea_case, basis, year, is_exploratory, path, capacity_2024_gw, capacity_target_gw, additions_total_gw, n_years, build_in_target_year_gw,
            build_first_year_gw, base_build_2023_24_gw, base_build_2020_24_avg_gw, growth_rate FROM mart_dc_capacity_path""")
save(dcp.round(4), "res_demand_dc_paths")

In [ ]:
# The two ranges of the 2030 total (EV transition + data-centre extra over the 2024 build), each as a percentage of 2025e world mine production
sens = rd("SELECT * FROM mart_demand_sensitivity")
ref = sens[sens.bar_order == 0].iloc[0]
lo1, hi1 = sens.loc[sens.headline_total_t.idxmin()], sens.loc[sens.headline_total_t.idxmax()]

ev30 = evc[(evc.year == 2030) & (evc.growth_case == "central")]
dc30 = dcx[dcx.year == 2030]
ev_lo, ev_hi = ev30.loc[ev30.transition_t.idxmin()], ev30.loc[ev30.transition_t.idxmax()]
dc_lo, dc_hi = dc30.loc[dc30.extra_vs_base_t.idxmin()], dc30.loc[dc30.extra_vs_base_t.idxmax()]
pathname = {"linear": "straight line", "constant_growth": "constant growth"}
basisname = {"Whole data centre": "total capacity", "IT equipment": "IT capacity"}


def corner(ev, dc):
    return (f"EV: IEA {ev.iea_scenario}, {pathname[ev.path_method]} to 2030, BEV {ev.bev_kg:g} kg, PHEV {ev.phev_kg:g} kg, petrol-car copper {ev.petrol_kg:g} kg ({ev.petrol_case.replace('_', ' ')}); "
            f"data centres: IEA {dc.iea_case}, {dc.path.replace('_', ' ')} build, {basisname[dc.basis]}, {dc.t_per_mw:g} t/MW, base-year build {dc.base_build_case}")


tot = lambda e, d: e.transition_t + d.extra_vs_base_t
ranges = pd.DataFrame([
    ("reference", "reference", ref.headline_total_t, f"Reference case (a convenience choice, not the most likely case): {ref.bar}"),
    ("one_at_a_time", "low", lo1.headline_total_t, f"{lo1.bar}: {lo1.alt_label}"),
    ("one_at_a_time", "high", hi1.headline_total_t, f"{hi1.bar}: {hi1.alt_label}"),
    ("all_combinations", "low", tot(ev_lo, dc_lo), corner(ev_lo, dc_lo)),
    ("all_combinations", "high", tot(ev_hi, dc_hi), corner(ev_hi, dc_hi)),
], columns=["range_id", "end", "total_t", "detail"])
ranges["total_kt"] = (ranges.total_t / 1000).round(1)
ranges["total_pct_of_mine"] = (ranges.total_t / MINE * 100).round(4)  # keep the decimals: 3.4456 must display as 3.4, not 3.5
ranges["cases_considered"] = [np.nan, len(sens), len(sens), len(ev30) * len(dc30), len(ev30) * len(dc30)]  # alternatives tried (one at a time) or combinations of all assumptions
ranges = ranges[["range_id", "end", "total_kt", "total_pct_of_mine", "cases_considered", "detail"]]
assert abs(ranges.loc[0, "total_pct_of_mine"] - ref.headline_total_pct_of_mine) < 0.0001
save(ranges, "res_demand_ranges")
print(ranges[["range_id", "end", "total_kt", "total_pct_of_mine"]].to_string(index=False))

## 3b. Copper-to-aluminium page: the ratio series and the facts about today
World Bank monthly prices (nominal US dollars per tonne, monthly averages), from January 1960. The ratio is copper divided by aluminium.
"Highest" means highest in this series. Facts are computed here so the website never calculates results itself.

In [ ]:
rs = rd("""SELECT month, copper_usd_t_avg AS copper_usd_t, aluminium_usd_t_avg AS aluminium_usd_t, cu_al_ratio AS ratio
           FROM mart_cu_al_ratio ORDER BY month""")
rs["ratio"] = rs["ratio"].round(4)
assert rs.notna().all().all() and rs.month.is_unique
save(rs, "res_ratio_series")
last = rs.iloc[-1]
before = rs.iloc[:-1]
earlier = rs.iloc[:-24]   # more than two years before the latest month
top = earlier.loc[earlier.ratio.idxmax()]
rf = pd.DataFrame([
    ("series_months", rs.month.iloc[0], len(rs), "months", "World Bank monthly copper and aluminium prices, January 1960 to the latest month"),
    ("ratio_latest", last.month, last.ratio, "ratio", "copper price divided by aluminium price, latest month"),
    ("copper_latest", last.month, last.copper_usd_t, "USD per tonne", "latest monthly average"),
    ("aluminium_latest", last.month, last.aluminium_usd_t, "USD per tonne", "latest monthly average"),
    ("ratio_highest_before_last_24m", top.month, top.ratio, "ratio", "highest ratio in the months before the latest 24 months"),
    ("months_below_latest", last.month, int((before.ratio < last.ratio).sum()), "months", "earlier months with a lower ratio than the latest month"),
    ("ratio_median_since_1970", "1970-01-01", rs[rs.month >= "1970-01-01"].ratio.median(), "ratio", "median of all months since January 1970"),
    ("ratio_min", rs.loc[rs.ratio.idxmin()].month, rs.ratio.min(), "ratio", "lowest ratio in the series"),
], columns=["fact_id", "month", "value", "unit", "note"])
rf["value"] = rf["value"].round(4)
save(rf, "res_ratio_facts")
print(rf[["fact_id", "month", "value"]].to_string(index=False))

## 3c. Dollar page: copper and the broad dollar index over time (the page's main chart)
World Bank monthly copper (USD per tonne, monthly average) and the FRED broad dollar index (monthly average of the daily values), from the first month
of the dollar index sample. Both are shown indexed to 100 in that first month, so a reader can compare their paths. The index is a level chart:
the statistics on the page use monthly changes, and the page says so.

In [ ]:
ds = rd("""SELECT month, copper_usd_t_avg AS copper_usd_t, dollar_index_avg AS dollar_index FROM mart_monthly_panel
           WHERE copper_usd_t_avg IS NOT NULL AND dollar_index_avg IS NOT NULL ORDER BY month""")
base = ds.iloc[0]
ds["copper_indexed"] = (ds.copper_usd_t / base.copper_usd_t * 100).round(2)
ds["dollar_indexed"] = (ds.dollar_index / base.dollar_index * 100).round(2)
ds["copper_usd_t"] = ds.copper_usd_t.round(0)
ds["dollar_index"] = ds.dollar_index.round(2)
assert ds.notna().all().all() and ds.month.is_unique
# the chart's sample must be the sample of the tables: one more month than the monthly changes
corr = pd.read_csv("../results/res_dollar_correlations.csv")
row = corr[corr.comparison.str.startswith("Copper vs broad dollar index")].iloc[0]
first_change = row.comparison.split(", ")[1].split(" to ")[0]
assert ds.month.iloc[1][:7] == first_change and len(ds) - 1 == row.months, "the series does not match the sample of the correlation table"
save(ds[["month", "copper_usd_t", "dollar_index", "copper_indexed", "dollar_indexed"]], "res_dollar_series")
print(ds.iloc[[0, -1]].to_string(index=False))

## 4. Data quality page: checks, sources, known issues, pipeline
Check details that mention LME data are withheld, except the LME-versus-World-Bank agreement (a correlation and a gap, no prices).

In [ ]:
chk = rd("SELECT check_no, table_name, description, status, detail, run_date FROM data_checks")
is_lme = chk.table_name.str.contains("lme", case=False) | chk.description.str.contains("lme", case=False)
keep = chk.table_name.eq("stg_lme_copper_daily vs stg_wb_prices_monthly")
chk.loc[is_lme & ~keep, "detail"] = "details withheld: the LME data is commercial and is never published"
save(chk, "res_dashboard_checks")
print(chk.status.value_counts().to_dict(), "| LME rows with details withheld:", int((is_lme & ~keep).sum()))

src = pd.read_csv("../copper_database/collected/sources.csv", keep_default_na=False, na_values=[""])
save(src[["source_id", "name", "publisher", "url", "source_type", "licence_note", "accessed_on", "reliability", "notes"]], "res_dashboard_sources")
ki = pd.read_csv("../copper_database/collected/known_issues.csv", keep_default_na=False, na_values=[""])
save(ki[["issue_id", "table_name", "description", "status", "source_id", "raised_on", "note"]], "res_dashboard_known_issues")
cat = rd("SELECT kind, COUNT(*) AS n_tables FROM tables_catalog GROUP BY kind ORDER BY kind")
save(cat, "res_dashboard_pipeline")

In [ ]:
con.commit()
con.close()
print("Written (rows, columns):")
for k, v in written.items():
    print(f"  {k}: {v[0]} rows, {v[1]} columns")